# Nucleotide Transformer with Real Genomic DNA

Tests Transformer scaling on **real genomic sequences** (human chr22).

## Purpose

- **Synthetic DNA (other notebook)**: Tests pure architectural properties
- **Real DNA (this notebook)**: Ecological validity for DNA Transformers

NT v2 models use 6-mer tokenization across 850 genomes. Testing on real
human DNA validates that the geometric tax is not an artifact of random sequences.

## Setup Instructions

1. Upload `evaluation_harness.py` and `perturbation_protocol.py`
2. Change Runtime to GPU (Runtime > Change runtime type > GPU)
3. Run cells in order

---

In [ ]:
# Install Dependencies

print("Installing core dependencies...")
!pip install -q torch transformers shesha-geometry matplotlib seaborn pandas einops

import transformers, torch
print(f"transformers {transformers.__version__}")
print(f"torch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# --- Patch transformers for Nucleotide Transformer compatibility ---
# NT v2 custom modeling code was written for transformers ~4.20-4.30.
# In transformers 5.x, several things were removed/moved:
#   1. find_pruneable_heads_and_indices / prune_linear_layer moved from pytorch_utils
#   2. PretrainedConfig no longer sets is_decoder, add_cross_attention, etc. by default
# This monkey-patch restores everything so trust_remote_code works.
import torch.nn as nn
import transformers.pytorch_utils as _ptu
from transformers import PretrainedConfig

# Patch 1: Restore missing functions in pytorch_utils
if not hasattr(_ptu, 'find_pruneable_heads_and_indices'):
    def find_pruneable_heads_and_indices(heads, n_heads, head_size, already_pruned_heads):
        mask = torch.ones(n_heads, head_size)
        heads = set(heads) - already_pruned_heads
        for head in heads:
            head = head - sum(1 if h < head else 0 for h in already_pruned_heads)
            mask[head] = 0
        mask = mask.view(-1).contiguous().eq(1)
        index = torch.arange(len(mask))[mask].long()
        return heads, index
    _ptu.find_pruneable_heads_and_indices = find_pruneable_heads_and_indices
    print("Patched: find_pruneable_heads_and_indices")

if not hasattr(_ptu, 'prune_linear_layer'):
    def prune_linear_layer(layer, index, dim=0):
        index = index.to(layer.weight.device)
        W = layer.weight.index_select(dim, index).clone().detach()
        if layer.bias is not None:
            b = layer.bias.clone().detach() if dim == 1 else layer.bias[index].clone().detach()
        new_size = list(layer.weight.size())
        new_size[dim] = len(index)
        new_layer = nn.Linear(new_size[1], new_size[0], bias=layer.bias is not None).to(layer.weight.device)
        new_layer.weight.requires_grad = False
        new_layer.weight.copy_(W.contiguous())
        new_layer.weight.requires_grad = True
        if layer.bias is not None:
            new_layer.bias.requires_grad = False
            new_layer.bias.copy_(b.contiguous())
            new_layer.bias.requires_grad = True
        return new_layer
    _ptu.prune_linear_layer = prune_linear_layer
    print("Patched: prune_linear_layer")

# Patch 2: Restore missing PreTrainedModel methods removed in transformers 5.x
# NT's custom EsmModel calls these during forward pass.
from transformers import PreTrainedModel as _PTM

if not hasattr(_PTM, 'get_head_mask'):
    def get_head_mask(self, head_mask, num_hidden_layers, is_attention_chunked=False):
        if head_mask is not None:
            if head_mask.dim() == 1:
                head_mask = head_mask.unsqueeze(0).unsqueeze(0).unsqueeze(-1).unsqueeze(-1)
                head_mask = head_mask.expand(num_hidden_layers, -1, -1, -1, -1)
            elif head_mask.dim() == 2:
                head_mask = head_mask.unsqueeze(1).unsqueeze(-1).unsqueeze(-1)
            head_mask = head_mask.to(dtype=next(self.parameters()).dtype)
            if is_attention_chunked:
                head_mask = head_mask.unsqueeze(-1)
        else:
            head_mask = [None] * num_hidden_layers
        return head_mask
    _PTM.get_head_mask = get_head_mask
    print("Patched: get_head_mask")

if not hasattr(_PTM, 'get_extended_attention_mask'):
    def get_extended_attention_mask(self, attention_mask, input_shape, device=None, dtype=None):
        if attention_mask.dim() == 3:
            extended = attention_mask[:, None, :, :]
        elif attention_mask.dim() == 2:
            extended = attention_mask[:, None, None, :]
        else:
            raise ValueError(f"Wrong shape for attention_mask: {attention_mask.shape}")
        if dtype is None:
            dtype = next(self.parameters()).dtype
        extended = extended.to(dtype=dtype)
        extended = (1.0 - extended) * torch.finfo(dtype).min
        return extended
    _PTM.get_extended_attention_mask = get_extended_attention_mask
    print("Patched: get_extended_attention_mask")

if not hasattr(_PTM, 'invert_attention_mask'):
    def invert_attention_mask(self, encoder_attention_mask):
        if encoder_attention_mask.dim() == 3:
            extended = encoder_attention_mask[:, None, :, :]
        elif encoder_attention_mask.dim() == 2:
            extended = encoder_attention_mask[:, None, None, :]
        else:
            raise ValueError(f"Wrong shape: {encoder_attention_mask.shape}")
        dtype = next(self.parameters()).dtype
        extended = extended.to(dtype=dtype)
        extended = (1.0 - extended) * torch.finfo(dtype).min
        return extended
    _PTM.invert_attention_mask = invert_attention_mask
    print("Patched: invert_attention_mask")

# Patch 3: Fix all_tied_weights_keys / tie_weights compatibility
# Same issue as Caduceus: newer transformers expects all_tied_weights_keys
# on all models, but NT's custom EsmForMaskedLM doesn't define it.
import transformers.modeling_utils as _mu
if not getattr(_mu, '_nt_patched', False):
    orig_mark = _mu.PreTrainedModel.mark_tied_weights_as_initialized
    def safe_mark(self):
        if not hasattr(self, 'all_tied_weights_keys'):
            self.all_tied_weights_keys = {}
        return orig_mark(self)
    _mu.PreTrainedModel.mark_tied_weights_as_initialized = safe_mark

    orig_finalize = _mu.PreTrainedModel._finalize_load_state_dict
    @staticmethod
    def safe_finalize(model, load_config, load_info):
        if not hasattr(model, 'all_tied_weights_keys'):
            model.all_tied_weights_keys = {}
        orig_tie = model.tie_weights
        def _patched_tie(missing_keys=None, recompute_mapping=False, **kwargs):
            return orig_tie()
        model.tie_weights = _patched_tie
        return orig_finalize(model, load_config, load_info)
    _mu.PreTrainedModel._finalize_load_state_dict = safe_finalize

    _mu._nt_patched = True
    print("Patched: all_tied_weights_keys / tie_weights")

# Patch 3: Restore missing default config attributes
# NT's custom EsmConfig inherits from PretrainedConfig, which used to set
# these in __init__. Transformers 5.x removed them, so we add them back
# as class-level defaults.
_config_defaults = {
    'is_decoder': False,
    'add_cross_attention': False,
    'chunk_size_feed_forward': 0,
    'is_encoder_decoder': False,
    'tie_word_embeddings': True,
}
for attr, default in _config_defaults.items():
    if not hasattr(PretrainedConfig, attr):
        setattr(PretrainedConfig, attr, default)
        print(f"Patched config default: {attr}={default}")

print("\nDone!")

In [ ]:
# Configuration

import sys, os
sys.path.insert(0, '.')
PHASE = 'full'

OUTPUT_BASE = './results/nt_realdna_experiment/'
RESULTS_DIR = OUTPUT_BASE + 'results'
CACHE_DIR = OUTPUT_BASE + 'cache'

CONFIG = {
    'quick': {
        'n_sequences': 500,
        'seq_length': 1000,
        'models': [
            'InstaDeepAI/nucleotide-transformer-v2-50m-multi-species',
            'InstaDeepAI/nucleotide-transformer-v2-500m-multi-species',
        ],
        'batch_size': 8,
        'n_bootstrap': 0,
        'snp_rates': [0.01, 0.05, 0.10],
    },
    'full': {
        'n_sequences': 10000,
        'seq_length': 1000,
        'models': [
            'InstaDeepAI/nucleotide-transformer-v2-50m-multi-species',
            'InstaDeepAI/nucleotide-transformer-v2-100m-multi-species',
            'InstaDeepAI/nucleotide-transformer-v2-250m-multi-species',
            'InstaDeepAI/nucleotide-transformer-v2-500m-multi-species',
            'InstaDeepAI/nucleotide-transformer-2.5b-multi-species',
        ],
        'batch_size': 8,
        'n_bootstrap': 5,
        'snp_rates': [0.01, 0.02, 0.05, 0.10],
    },
}

BATCH_OVERRIDES = {
    'nucleotide-transformer-2.5b-multi-species': 1,
    'nucleotide-transformer-v2-500m-multi-species': 2,
}

config = CONFIG[PHASE]
print(f"Phase: {PHASE.upper()}")
print(f"Architecture: Nucleotide Transformer (DNA Transformer)")
print(f"Data: Real human genomic DNA (chr22)")
print(f"Sequences: {config['n_sequences']}")
print(f"Models: {len(config['models'])}")

In [ ]:
# Load Real Genomic DNA

import urllib.request
import gzip
import os
import numpy as np

CHR22_URL = 'https://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/chr22.fa.gz'
VALID_BASES = set('ACGT')


def download_and_sample_genomic_dna(n_sequences, seq_length, seed=320):
    cache_file = f'{CACHE_DIR}/chr22_sample_{n_sequences}_{seq_length}.txt'

    if os.path.exists(cache_file):
        print(f"Loading cached genomic sequences: {cache_file}")
        with open(cache_file) as f:
            sequences = [line.strip() for line in f if line.strip()]
        print(f"Loaded {len(sequences)} cached sequences")
        return sequences

    print("Downloading human chr22 (~50MB, first run only)...")
    os.makedirs(CACHE_DIR, exist_ok=True)

    with urllib.request.urlopen(CHR22_URL) as response:
        with gzip.GzipFile(fileobj=response) as f:
            lines = f.read().decode('utf-8').split('\n')
            chr22_seq = ''.join(line.strip() for line in lines[1:] if line.strip())

    print(f"Downloaded chr22 ({len(chr22_seq):,} bp)")

    buffer_factor = 1.5
    max_attempts = int(n_sequences * buffer_factor)
    rng = np.random.default_rng(seed)
    sequences = []

    for attempt in range(max_attempts):
        start = rng.integers(0, len(chr22_seq) - seq_length)
        seq = chr22_seq[start:start + seq_length].upper()

        n_count = sum(1 for c in seq if c not in VALID_BASES)
        if n_count < seq_length * 0.1:
            seq_clean = ''.join(
                c if c in VALID_BASES else rng.choice(['A', 'C', 'G', 'T'])
                for c in seq
            )
            sequences.append(seq_clean)

        if len(sequences) >= n_sequences:
            break

    sequences = sequences[:n_sequences]
    print(f"Sampled {len(sequences)} clean sequences")

    with open(cache_file, 'w') as f:
        f.write('\n'.join(sequences))
    print(f"Cached to {cache_file}")
    return sequences


sequences = download_and_sample_genomic_dna(
    config['n_sequences'], config['seq_length'], seed=320,
)

print(f"\nSequence stats:")
print(f"  Count: {len(sequences)}")
print(f"  Length: {len(sequences[0])}")
print(f"  Sample: {sequences[0][:60]}...")

total_bases = sum(len(s) for s in sequences)
print(f"\nBase composition:")
for base in 'ACGT':
    count = sum(s.count(base) for s in sequences)
    print(f"  {base}: {count/total_bases*100:.1f}%")

In [ ]:
# DNA Perturbation Suite

from dataclasses import dataclass, field

DNA_BASES = ['A', 'C', 'G', 'T']
COMPLEMENT = {'A': 'T', 'T': 'A', 'C': 'G', 'G': 'C'}

@dataclass
class PerturbedSet:
    name: str
    category: str
    sequences: list
    params: dict = field(default_factory=dict)
    description: str = ''


def mutate_dna(sequence, mutation_rate, rng):
    """SNP mutations. Only mutates valid ACGT bases."""
    seq = list(sequence)
    valid_positions = [i for i, c in enumerate(seq) if c in DNA_BASES]
    n_mutations = max(1, int(len(valid_positions) * mutation_rate))
    positions = rng.choice(valid_positions, size=min(n_mutations, len(valid_positions)), replace=False)
    for pos in positions:
        original = seq[pos]
        alternatives = [b for b in DNA_BASES if b != original]
        seq[pos] = rng.choice(alternatives)
    return ''.join(seq)


def reverse_complement(sequence):
    return ''.join(COMPLEMENT.get(b, b) for b in reversed(sequence))


class DNAPerturbationSuite:
    def __init__(self, seed=320, snp_rates=None, include_rc=True):
        self.rng = np.random.default_rng(seed)
        self.snp_rates = snp_rates or [0.01, 0.02, 0.05, 0.10]
        self.include_rc = include_rc

    def run_all(self, sequences):
        results = {}
        for rate in self.snp_rates:
            name = f"snp_{int(rate * 100)}pct"
            perturbed = [mutate_dna(s, rate, self.rng) for s in sequences]
            results[name] = PerturbedSet(
                name=name, category='snp', sequences=perturbed,
                params={'mutation_rate': rate},
                description=f'SNP mutations at {rate*100:.0f}% rate',
            )
        if self.include_rc:
            results['reverse_complement'] = PerturbedSet(
                name='reverse_complement', category='rc',
                sequences=[reverse_complement(s) for s in sequences],
                params={}, description='Reverse complement transformation',
            )
        return results


suite = DNAPerturbationSuite(seed=320, snp_rates=config['snp_rates'])

test_perturbed = suite.run_all(sequences[:3])
for name, pset in test_perturbed.items():
    dists = [
        sum(a != b for a, b in zip(o, p)) / len(o)
        for o, p in zip(sequences[:3], pset.sequences)
    ]
    print(f"  {name}: mean_hamming={np.mean(dists):.4f}")
print("Perturbation suite ready")

In [ ]:
# Nucleotide Transformer Model Wrapper

import torch
import gc
from transformers import AutoTokenizer, AutoModelForMaskedLM


def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        mem = torch.cuda.memory_allocated() / 1e9
        print(f"  GPU memory after cleanup: {mem:.2f} GB")


def make_nt_embedding_fn(model_name, batch_size=4):
    """Load Nucleotide Transformer and return embedding function."""
    print(f"Loading {model_name}...")
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"  Device: {device}")

    short = model_name.split('/')[-1]
    effective_batch = BATCH_OVERRIDES.get(short, batch_size)
    if effective_batch != batch_size:
        print(f"  Batch size override: {batch_size} -> {effective_batch}")

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    model = AutoModelForMaskedLM.from_pretrained(
        model_name, trust_remote_code=True,
    ).to(device).eval()

    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    if torch.cuda.is_available():
        mem = torch.cuda.memory_allocated() / 1e9
        print(f"  Params: {n_params:.1f}M | GPU mem: {mem:.2f} GB")
    else:
        print(f"  Params: {n_params:.1f}M")

    max_length = tokenizer.model_max_length
    if max_length > 100000:
        max_length = 1024
    print(f"  Max token length: {max_length}")

    @torch.no_grad()
    def embed(sequences):
        embeddings = []
        n_batches = (len(sequences) + effective_batch - 1) // effective_batch

        for i in range(0, len(sequences), effective_batch):
            batch = sequences[i:i + effective_batch]
            batch_idx = i // effective_batch + 1

            if batch_idx % 20 == 0 or batch_idx == n_batches:
                print(f"    Batch {batch_idx}/{n_batches}", end='\r')

            tokens = tokenizer(
                batch, return_tensors='pt', padding=True,
                truncation=True, max_length=max_length,
            )
            tokens = {k: v.to(device) for k, v in tokens.items()}

            outputs = model(**tokens, output_hidden_states=True)
            hidden = outputs.hidden_states[-1]

            attention_mask = tokens['attention_mask'].unsqueeze(-1)
            pooled = (hidden * attention_mask).sum(dim=1) / attention_mask.sum(dim=1).clamp(min=1)
            embeddings.append(pooled.cpu().numpy())

        print()
        return np.concatenate(embeddings, axis=0)

    return embed, model, tokenizer, n_params

print("Nucleotide Transformer wrapper ready")

In [ ]:
# Evaluation Harness

from evaluation_harness import StabilityHarness

harness = StabilityHarness(
    window_size=0,
    metric='cosine',
    n_splits=30,
    seed=320,
    max_samples=2500,
    n_bootstrap=config['n_bootstrap'],
)

print(f"Harness configured (bootstrap={config['n_bootstrap']})")

In [ ]:
# Run Experiment

import os
import time
import pandas as pd
from evaluation_harness import ModelReport

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

print('=' * 70)
print(f'NUCLEOTIDE TRANSFORMER + REAL GENOMIC DNA -- Phase: {PHASE.upper()}')
print('=' * 70)

reports = []
model_param_counts = []
all_detailed_rows = []

for model_idx, model_name in enumerate(config['models']):
    print(f"\n{'='*70}")
    print(f"[{model_idx+1}/{len(config['models'])}] {model_name}")
    print('=' * 70)

    start_time = time.time()

    embed_fn, model_obj, tokenizer_obj, n_params_m = make_nt_embedding_fn(
        model_name, config['batch_size']
    )
    model_param_counts.append(n_params_m)

    print('\nGenerating perturbations...')
    perturbed_sets = suite.run_all(sequences)

    safe_name = model_name.replace('/', '_').replace('-', '_') + '_realdna'
    cache_clean = f'{CACHE_DIR}/{safe_name}_clean.npy'

    if os.path.exists(cache_clean):
        print(f'Loading cached: {cache_clean}')
        embeddings_clean = np.load(cache_clean)
    else:
        print(f'Computing clean embeddings ({len(sequences)} sequences)...')
        embeddings_clean = embed_fn(sequences)
        np.save(cache_clean, embeddings_clean)
        print(f'  Cached to {cache_clean}')
    print(f'  Shape: {embeddings_clean.shape}')

    perturbed_embeddings = {}
    for pert_name, pset in perturbed_sets.items():
        cache_pert = f'{CACHE_DIR}/{safe_name}_{pert_name}.npy'
        if os.path.exists(cache_pert):
            print(f'  Loading cached: {pert_name}')
            perturbed_embeddings[pert_name] = np.load(cache_pert)
        else:
            print(f'  Embedding: {pert_name}...')
            perturbed_embeddings[pert_name] = embed_fn(pset.sequences)
            np.save(cache_pert, perturbed_embeddings[pert_name])

    del model_obj, tokenizer_obj
    cleanup_gpu()

    print('\nRunning Shesha evaluation...')
    report = harness.evaluate_all_perturbations(
        model_name=model_name,
        embeddings_clean=embeddings_clean,
        perturbed_dict=perturbed_embeddings,
    )
    reports.append(report)

    short_name = model_name.split('/')[-1]
    for pert_name, metrics in report.perturbation_breakdown().items():
        all_detailed_rows.append({
            'model': short_name,
            'size_M': round(n_params_m, 1),
            'perturbation': pert_name,
            'rdm_similarity': metrics.get('rdm_similarity_score', 0),
            'rdm_drift': metrics.get('rdm_drift', 0),
            'pert_stability': metrics.get('perturbation_stability_score', 0),
            'pert_magnitude': metrics.get('perturbation_magnitude', 0),
            'composite': metrics.get('composite_stability', 0),
        })

    elapsed = time.time() - start_time
    summary = report.summary()
    print(f'\nCompleted in {elapsed/60:.1f} min')
    print(f'  Composite Stability: {summary["mean_composite_stability"]:.4f}')
    print(f'  RDM Similarity:      {summary["mean_rdm_similarity_score"]:.4f}')

df_detailed = pd.DataFrame(all_detailed_rows)
detailed_path = f'{RESULTS_DIR}/nt_realdna_{PHASE}_detailed.csv'
df_detailed.to_csv(detailed_path, index=False)
print(f'\nPer-perturbation results:')
print(df_detailed.to_string(index=False))
print(f'\nSaved to {detailed_path}')

print('\n' + '=' * 70)
print('EXPERIMENT COMPLETE')
print('=' * 70)

In [ ]:
# Results & Visualization

from evaluation_harness import compare_models
import matplotlib.pyplot as plt

comparison = compare_models(reports, output_dir=RESULTS_DIR)

print('\n' + '=' * 70)
print('NT + REAL DNA SUMMARY')
print('=' * 70 + '\n')

for model_name, data in comparison['models'].items():
    s = data['summary']
    short = model_name.split('/')[-1]
    print(f'{short}:')
    print(f'  Composite Stability:  {s["mean_composite_stability"]:.4f} +/- {s["std_composite_stability"]:.4f}')
    print()

summaries = [r.summary() for r in reports]
values = [s['mean_composite_stability'] for s in summaries]

plt.figure(figsize=(10, 6))
plt.plot(model_param_counts, values, 'o-', linewidth=2, markersize=10, color='tab:orange')
plt.xlabel('Parameters (M)')
plt.ylabel('Composite Stability')
plt.title('Nucleotide Transformer on Real Genomic DNA: Stability vs. Size', fontweight='bold')
if max(model_param_counts) / max(min(model_param_counts), 0.01) > 10:
    plt.xscale('log')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/nt_realdna_scaling_{PHASE}.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved to {RESULTS_DIR}/nt_realdna_scaling_{PHASE}.png')

In [ ]:
# Cross-Experiment Comparison

import matplotlib.pyplot as plt
import pandas as pd
import glob

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel 1: NT synthetic vs real DNA
ax1 = axes[0]
nt_syn_files = glob.glob(f'{RESULTS_DIR}/../../../**/nt_scaling_*_detailed.csv', recursive=True)
if nt_syn_files:
    df_syn = pd.read_csv(nt_syn_files[0])
    syn_agg = df_syn.groupby('size_M')['composite'].mean().reset_index()
    ax1.plot(syn_agg['size_M'], syn_agg['composite'], 'o--',
             color='tab:gray', linewidth=2, markersize=10, label='NT (Synthetic DNA)', alpha=0.6)

real_agg = df_detailed.groupby('size_M')['composite'].mean().reset_index()
ax1.plot(real_agg['size_M'], real_agg['composite'], 's-',
         color='tab:orange', linewidth=2, markersize=10, label='NT (Real DNA)')

ax1.set_xlabel('Parameters (M)', fontsize=12)
ax1.set_ylabel('Composite Stability', fontsize=12)
ax1.set_title('NT: Synthetic vs Real DNA', fontweight='bold')
if len(real_agg) > 1 and real_agg['size_M'].max() / real_agg['size_M'].min() > 10:
    ax1.set_xscale('log')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Panel 2: All architectures
ax2 = axes[1]
ax2.plot(real_agg['size_M'], real_agg['composite'], 's-',
         color='tab:orange', linewidth=2, markersize=10, label='NT (Transformer)')

esm2_files = glob.glob(f'{RESULTS_DIR}/../../../**/esm2_*_detailed.csv', recursive=True)
if esm2_files:
    df_esm2 = pd.read_csv(esm2_files[0])
    esm2_agg = df_esm2.groupby('size_M')['composite'].mean().reset_index()
    ax2.plot(esm2_agg['size_M'], esm2_agg['composite'], 'o-',
             color='tab:red', linewidth=2, markersize=10, label='ESM-2 (Transformer)')

cad_files = glob.glob(f'{RESULTS_DIR}/../../../**/caduceus_*_detailed.csv', recursive=True)
if cad_files:
    df_cad = pd.read_csv(cad_files[0])
    cad_agg = df_cad.groupby('size_M')['composite'].mean().reset_index()
    ax2.plot(cad_agg['size_M'], cad_agg['composite'], 'D-',
             color='tab:blue', linewidth=2, markersize=10, label='Caduceus (SSM)')

ax2.set_xscale('log')
ax2.set_xlabel('Parameters (M)', fontsize=12)
ax2.set_ylabel('Composite Stability', fontsize=12)
ax2.set_title('All Architectures: The Geometric Tax', fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/nt_realdna_crossexp_{PHASE}.png', dpi=300, bbox_inches='tight')
plt.show()